### Code funzionante

In [145]:
a = 1000
b = 432

lut = [(1 << 32) // ((1 << 15) | (i << 8)) for i in range(32)]

s = 16 - b.bit_length()
b_norm = b << s

idx = (b_norm >> 8) & 0xF

x = lut[idx] 

for _ in range(2):
    x = (x * ((2 << 32) - b_norm * x)) >> 32

result = (a * x) >> (32 - s)

print(result)
print(a // b)  # esatto

2
2


### Todo: map the LUT as a polynomial

In [146]:
def get_bit(n, i):
    return (n >> i) & 1

In [147]:
import numpy as np
import math

def polylut(bit = 0):
    N = 32
    seq = np.array([get_bit(lut[i], bit) for i in range(N)], dtype=float)
    c = np.fft.fft(seq)  

    def f(x):
        s = 0.0
        for k in range(N):
            ang = 2 * math.pi * k * x / N
            s += c[k].real * math.cos(ang) - c[k].imag * math.sin(ang)
        return s / N

    return f

f = polylut(0)

# Verifica sui punti di interpolazione
print("Verifica interpolazione:")
for x in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    val = f(x)
    expected = polylut(0)(x)
    print(f"  x={x:3d}: f(x)={val:.6f}, f(x)={expected:.6f}, err={abs(val-expected):.2e}")

Verifica interpolazione:
  x=  0: f(x)=0.000000, f(x)=0.000000, err=0.00e+00
  x=  1: f(x)=1.000000, f(x)=1.000000, err=0.00e+00
  x=  2: f(x)=1.000000, f(x)=1.000000, err=0.00e+00
  x=  3: f(x)=-0.000000, f(x)=-0.000000, err=0.00e+00
  x=  4: f(x)=0.000000, f(x)=0.000000, err=0.00e+00
  x=  5: f(x)=0.000000, f(x)=0.000000, err=0.00e+00
  x=  6: f(x)=1.000000, f(x)=1.000000, err=0.00e+00
  x=  7: f(x)=1.000000, f(x)=1.000000, err=0.00e+00
  x=  8: f(x)=1.000000, f(x)=1.000000, err=0.00e+00
  x=  9: f(x)=1.000000, f(x)=1.000000, err=0.00e+00
  x= 10: f(x)=-0.000000, f(x)=-0.000000, err=0.00e+00


In [148]:
import math
import numpy as np
import matplotlib.pyplot as plt
from random import randrange
import random

class Chebyshev:
    """
    Chebyshev(a, b, n, func)
    Given a function func, lower and upper limits of the interval [a,b],
    and maximum degree n, this class computes a Chebyshev approximation
    of the function.
    Method eval(x) yields the approximated function value.
    """
    def __init__(self, a, b, n, func):
        n = n + 1
        self.a = a
        self.b = b
        self.func = func

        bma = 0.5 * (b - a)
        bpa = 0.5 * (b + a)
        f = [func(math.cos(math.pi * (k + 0.5) / n) * bma + bpa) for k in range(n)]
        self.roots = f

        self.x = [math.cos(math.pi * (k + 0.5) / n) * bma + bpa for k in range(n)]

        fac = 2.0 / n
        self.c = [fac * sum([f[k] * math.cos(math.pi * j * (k + 0.5) / n)
                  for k in range(n)]) for j in range(n)]

    def eval(self, x):
        a,b = self.a, self.b
        #assert(a <= x <= b)
        y = (2.0 * x - a - b) * (1.0 / (b - a))
        y2 = 2.0 * y
        (d, dd) = (self.c[-1], 0)             # Special case first step for efficiency
        for cj in self.c[-2:0:-1]:            # Clenshaw's recurrence
            (d, dd) = (y2 * d - dd + cj, d)
        return y * d - dd + 0.5 * self.c[0]   # Last step is different

In [150]:
polyluts = []

for bit in range(18):
    f = polylut(bit)
    
    c = Chebyshev(0, 128, 445, f)
    
    polyluts.append(c)

### Step 1: doing the same as above but with poly

In [151]:
a = 1000
b = 432

lut = [(1 << 32) // ((1 << 15) | (i << 8)) for i in range(32)]

s = 16 - b.bit_length()
b_norm = b << s

idx = (b_norm >> 8) & 0xF

idx

8

In [153]:
binx = []

for i in range(32):
    if i >= len(polyluts):
        binx.append(0)
    else:
        binx.append(polyluts[i].eval(idx))

In [156]:
polydec = ([round(x) for x in list(binx)][::-1])

In [159]:
n = int("".join(map(str, polydec)), 2)

In [169]:
x = n

for _ in range(2):
    term = b_norm * x #Mult
    term = 2**33 - term  #Implementato come !term + 1, dove i bit più alti di 33 sono a 0
    
    x = x * term #Mult normale
    x = x >> 32 #Shift gratis

result = (a * x) >> (32 - s)
result

2